# Làm sạch dữ liệu BI10 Vòng 2

Notebook này đọc 5 file CSV gốc, áp dụng các bước làm sạch mà nhóm đã thống nhất sau phần EDA, rồi xuất 5 file CSV sạch vào thư mục `clean_data`. Mọi phân tích sâu hơn (Financial Health, Engagement, phân khúc) dùng các file trong `clean_data`, không dùng lại file gốc, để số liệu của cả nhóm khớp nhau.

Mỗi bước đều in số dòng hoặc ví dụ trước và sau khi xử lý để kiểm tra được. Lý do của từng thay đổi được ghi trong `clean_data/README.md`.

Cài thư viện: `pip install duckdb pandas`. Notebook đặt trong thư mục `BI10_Round2`, cùng cấp với `BI10_ROUND02_DATASET`.

In [ ]:
import os, re, unicodedata
import duckdb, pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

DATA_DIR = 'BI10_ROUND02_DATASET'
OUT_DIR = 'clean_data'
SNAPSHOT = '2025-06-04'          # ngày cuối của CONTRACT, LEADS, PAST_BEHAVIOR
os.makedirs(OUT_DIR, exist_ok=True)

con = duckdb.connect('bi10_eda.duckdb')   # dùng chung file với notebook EDA, chỉ nạp CSV nếu chưa có
con.execute("SET memory_limit='2GB'")
con.execute("SET preserve_insertion_order=false")

def q(sql):
    return con.sql(sql).df()

existing = set(q("select table_name from information_schema.tables")['table_name'])
for t in ['CUSTOMER', 'CONTRACT', 'LEADS', 'PAST_BEHAVIOR', 'CALL']:
    if t not in existing:
        con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv('{DATA_DIR}/{t}.csv', header=true, sample_size=-1)")
print('Đã có đủ 5 bảng gốc')

Đã có đủ 5 bảng gốc


## 1. Bảng quy đổi mã sản phẩm

Mỗi bảng dùng một bộ mã sản phẩm riêng. Cột mới `PRODUCT_GROUP` quy về 5 nhóm của đề: CD, TW, BNPL, CC, CL. Các mã chưa rõ nghĩa (CW, IN, EPOS, SAI) giữ nguyên mã gốc để không gán sai; XNA và ô trống để trống. Quy đổi này là giả định của nhóm vì đề không có bảng đối chiếu.

In [ ]:
PRODUCT_MAP = {
    # CONTRACT
    'CD': 'CD', 'TW': 'TW', 'BNPL': 'BNPL', 'CC': 'CC', 'CL': 'CL', 'CW': 'CW', 'IN': 'IN',
    # LEADS
    'BNP': 'BNPL', 'CLX': 'CL', 'ACL': 'CL', 'CCX': 'CC', 'VCC': 'CC', 'SCW': 'CC', 'CCW': 'CC', 'EPOS': 'EPOS',
    # PAST_BEHAVIOR
    'ACLX': 'CL', 'CC_SC': 'CC', 'CC_CC': 'CC', 'CC_VCC': 'CC', 'SAI': 'SAI',
    # CALL
    'ACLD': 'CL',
}
product_map = pd.DataFrame(list(PRODUCT_MAP.items()), columns=['CODE', 'PRODUCT_GROUP'])
con.register('product_map_df', product_map)
con.execute('CREATE OR REPLACE TABLE PRODUCT_MAP AS SELECT * FROM product_map_df')
product_map

,CODE,PRODUCT_GROUP
0,CD,CD
1,TW,TW
2,BNPL,BNPL
3,CC,CC
4,CL,CL
5,CW,CW
6,IN,IN
7,BNP,BNPL
8,CLX,CL
9,ACL,CL


## 2. Bảng quy đổi tỉnh

Địa chỉ được ghi theo dạng "phường, quận, tỉnh". Theo yêu cầu của nhóm, chỉ giữ tỉnh và quy về đúng 63 tỉnh thành (theo danh mục trước khi sáp nhập năm 2025, khớp với cách ghi trong dữ liệu). Cách làm: lấy phần sau dấu phẩy cuối cùng, bỏ dấu, bỏ chữ "TP.", "Thành phố", "Tỉnh", bỏ khoảng trắng và dấu chấm, rồi so với danh sách 63 tỉnh.

In [ ]:
PROVINCES = {
    'An Giang': ['angiang'], 'Bà Rịa - Vũng Tàu': ['bariavungtau', 'brvt', 'vungtau', 'bariavt'],
    'Bạc Liêu': ['baclieu'], 'Bắc Giang': ['bacgiang'], 'Bắc Kạn': ['backan', 'baccan'], 'Bắc Ninh': ['bacninh'],
    'Bến Tre': ['bentre'], 'Bình Định': ['binhdinh'], 'Bình Dương': ['binhduong'], 'Bình Phước': ['binhphuoc'],
    'Bình Thuận': ['binhthuan'], 'Cà Mau': ['camau'], 'Cao Bằng': ['caobang'], 'Cần Thơ': ['cantho'],
    'Đà Nẵng': ['danang'], 'Đắk Lắk': ['daklak', 'daclac', 'daklac', 'daclak'], 'Đắk Nông': ['daknong', 'dacnong'],
    'Điện Biên': ['dienbien'], 'Đồng Nai': ['dongnai'], 'Đồng Tháp': ['dongthap'], 'Gia Lai': ['gialai'],
    'Hà Giang': ['hagiang'], 'Hà Nam': ['hanam'], 'Hà Nội': ['hanoi'], 'Hà Tĩnh': ['hatinh'],
    'Hải Dương': ['haiduong'], 'Hải Phòng': ['haiphong'], 'Hậu Giang': ['haugiang'], 'Hòa Bình': ['hoabinh'],
    'Hưng Yên': ['hungyen'], 'Khánh Hòa': ['khanhhoa'], 'Kiên Giang': ['kiengiang'], 'Kon Tum': ['kontum'],
    'Lai Châu': ['laichau'], 'Lâm Đồng': ['lamdong'], 'Lạng Sơn': ['langson'], 'Lào Cai': ['laocai'],
    'Long An': ['longan'], 'Nam Định': ['namdinh'], 'Nghệ An': ['nghean'], 'Ninh Bình': ['ninhbinh'],
    'Ninh Thuận': ['ninhthuan'], 'Phú Thọ': ['phutho'], 'Phú Yên': ['phuyen'], 'Quảng Bình': ['quangbinh'],
    'Quảng Nam': ['quangnam'], 'Quảng Ngãi': ['quangngai'], 'Quảng Ninh': ['quangninh'], 'Quảng Trị': ['quangtri'],
    'Sóc Trăng': ['soctrang'], 'Sơn La': ['sonla'], 'Tây Ninh': ['tayninh'], 'Thái Bình': ['thaibinh'],
    'Thái Nguyên': ['thainguyen'], 'Thanh Hóa': ['thanhhoa'], 'Thừa Thiên Huế': ['thuathienhue', 'hue', 'tthue'],
    'Tiền Giang': ['tiengiang'], 'Hồ Chí Minh': ['hcm', 'hochiminh', 'saigon'], 'Trà Vinh': ['travinh'],
    'Tuyên Quang': ['tuyenquang'], 'Vĩnh Long': ['vinhlong'], 'Vĩnh Phúc': ['vinhphuc'], 'Yên Bái': ['yenbai'],
}
assert len(PROVINCES) == 63
KEY_TO_PROVINCE = {k: p for p, keys in PROVINCES.items() for k in keys}

def norm(s):
    s = unicodedata.normalize('NFD', s)
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn').replace('đ', 'd').replace('Đ', 'D').lower().strip()
    s = re.sub(r'^(thanh pho|tp\.?|tinh)\s*', '', s)
    return re.sub(r'[^a-z]', '', s)

raw = q("select trim(split_part(ADDRESS, ',', -1)) RAW_PROVINCE, count(*) SO_KHACH from CUSTOMER where ADDRESS is not null group by 1")
raw['PROVINCE'] = raw.RAW_PROVINCE.map(lambda x: KEY_TO_PROVINCE.get(norm(x)) if isinstance(x, str) else None)
con.register('prov_df', raw[['RAW_PROVINCE', 'PROVINCE']])
con.execute('CREATE OR REPLACE TABLE PROVINCE_MAP AS SELECT * FROM prov_df')
print('Số cách ghi gốc:', len(raw), '| Số tỉnh sau quy đổi:', raw.PROVINCE.nunique())
print('Cách ghi không khớp tỉnh nào:')
display(raw[raw.PROVINCE.isna()].sort_values('SO_KHACH', ascending=False))
raw.sort_values('SO_KHACH', ascending=False).head(15)

Số cách ghi gốc: 68 | Số tỉnh sau quy đổi: 63
Cách ghi không khớp tỉnh nào:


,RAW_PROVINCE,SO_KHACH,PROVINCE
67,#NAME?,1,None


,RAW_PROVINCE,SO_KHACH,PROVINCE
43,TP. HCM,100528,Hồ Chí Minh
48,Dong Nai,39529,Đồng Nai
6,An Giang,34346,An Giang
7,TP. Ha Noi,33940,Hà Nội
8,Binh Duong,33550,Bình Dương
14,Long An,28143,Long An
51,Kien Giang,25488,Kiên Giang
17,Dong Thap,23469,Đồng Tháp
15,Tien Giang,23361,Tiền Giang
4,Thanh Hoa,22140,Thanh Hóa


Những cách ghi không khớp tỉnh nào ở trên (nếu có) sẽ để trống tỉnh. Cần nhìn danh sách này: nếu có cách ghi rõ ràng là một tỉnh mà bị bỏ sót, thêm vào `PROVINCES` rồi chạy lại.

## 3. CUSTOMER

Các thay đổi:

- Bỏ `FLAG_CAR_OWNER` (100% là "X"), `NAME_SALARY_FREQUENCY` (99% XNA), `CNT_PERSON_DEPENDENT` (98,5% trống).
- Thay `ADDRESS` bằng `PROVINCE` theo bảng quy đổi ở mục 2.
- Thêm `AGE_AT_LAST_APPLY`: tuổi lúc nộp hồ sơ gần nhất. Khách có tuổi lúc nộp hồ sơ dưới 18 hoặc trên 80 được coi là sai ngày sinh: để trống cả `DATE_BIRTH` và tuổi, đánh dấu `FLAG_INVALID_BIRTH = 1`.
- Giữ nguyên `CNT_CHILDREN`, kể cả các giá trị trên 5.

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE CL_CUSTOMER AS
WITH last_apply AS (SELECT SKP_CLIENT, max(APPLY_CONTRACT_TIME) LAST_APPLY FROM CONTRACT GROUP BY 1),
a AS (
  SELECT c.*, p.PROVINCE,
         floor(date_diff('day', c.DATE_BIRTH, l.LAST_APPLY) / 365.25)::int AGE_RAW
  FROM CUSTOMER c
  LEFT JOIN last_apply l USING (SKP_CLIENT)
  LEFT JOIN PROVINCE_MAP p ON trim(split_part(c.ADDRESS, ',', -1)) = p.RAW_PROVINCE
)
SELECT SKP_CLIENT, NAME_GENDER, NAME_EDUCATION_TYPE,
       CASE WHEN AGE_RAW BETWEEN 18 AND 80 THEN DATE_BIRTH::date END AS DATE_BIRTH,
       CASE WHEN AGE_RAW BETWEEN 18 AND 80 THEN AGE_RAW END AS AGE_AT_LAST_APPLY,
       CASE WHEN AGE_RAW BETWEEN 18 AND 80 THEN 0 ELSE 1 END AS FLAG_INVALID_BIRTH,
       CNT_CHILDREN, PROVINCE, AVG_SESSION_PER_WEEK_2025
FROM a
''')
q('''
select count(*) so_khach, count(distinct SKP_CLIENT) so_khach_khong_trung,
       sum(FLAG_INVALID_BIRTH) so_khach_sai_ngay_sinh,
       count(PROVINCE) co_tinh, count(*) - count(PROVINCE) trong_tinh,
       count(distinct PROVINCE) so_tinh, max(CNT_CHILDREN) so_con_lon_nhat
from CL_CUSTOMER
''')

,so_khach,so_khach_khong_trung,so_khach_sai_ngay_sinh,co_tinh,trong_tinh,so_tinh,so_con_lon_nhat
0,1000528,1000528,32.0,851285,149243,63,99


## 4. CONTRACT

Các thay đổi:

- Khử trùng: mỗi `SKP_CREDIT_CASE` giữ một dòng, ưu tiên dòng có học vấn, nghề nghiệp, loại thu nhập khác XNA. EDA đã kiểm tra rằng các dòng trùng giống nhau ở mọi trường nghiệp vụ, chỉ khác phần thông tin cá nhân.
- Thêm `IS_SIGNED` (1 = Active, Finished, Paid off, Signed, Written off; 0 = các trạng thái khác) theo định nghĩa của đề.
- Thêm `AMT_CREDIT_ADJ`: bằng `AMT_BILLING` với BNPL (vì `AMT_CREDIT` của BNPL luôn bằng 0), bằng `AMT_CREDIT` với các sản phẩm khác.
- Thêm `PRODUCT_GROUP` theo bảng quy đổi ở mục 1.
- Thêm `AGE_AT_APPLY`: tuổi của khách tại ngày nộp chính hợp đồng đó, tính từ ngày sinh đã làm sạch.
- Giữ nguyên thu nhập, kể cả các giá trị rất lớn như 9 tỷ.

In [ ]:
before = q('select count(*) so_dong, count(distinct SKP_CREDIT_CASE) so_hop_dong from CONTRACT')
con.execute('''
CREATE OR REPLACE TABLE CL_CONTRACT AS
WITH d AS (
  SELECT * FROM CONTRACT
  QUALIFY row_number() OVER (PARTITION BY SKP_CREDIT_CASE
          ORDER BY (NAME_EDUCATION_TYPE = 'XNA' AND NAME_INCOME_TYPE = 'XNA' AND CODE_PROFESSION = 'XNA'), AMT_INCOME_MAIN DESC) = 1
)
SELECT d.*,
       CASE WHEN d.NAME_CREDIT_STATUS IN ('Active','Finished','Paid off','Signed','Written off') THEN 1 ELSE 0 END AS IS_SIGNED,
       CASE WHEN d.PRODUCT = 'BNPL' THEN d.AMT_BILLING ELSE d.AMT_CREDIT END AS AMT_CREDIT_ADJ,
       m.PRODUCT_GROUP,
       floor(date_diff('day', c.DATE_BIRTH, d.APPLY_CONTRACT_TIME) / 365.25)::int AS AGE_AT_APPLY
FROM d
LEFT JOIN PRODUCT_MAP m ON d.PRODUCT = m.CODE
LEFT JOIN CL_CUSTOMER c USING (SKP_CLIENT)
''')
after = q('select count(*) so_dong, count(distinct SKP_CREDIT_CASE) so_hop_dong from CL_CONTRACT')
display(pd.concat([before.assign(bang='CONTRACT gốc'), after.assign(bang='CL_CONTRACT')]).set_index('bang'))
q('''
select PRODUCT, PRODUCT_GROUP, count(*) so_hd, sum(IS_SIGNED) da_ky,
       median(AMT_CREDIT) trung_vi_amt_credit, median(AMT_CREDIT_ADJ) trung_vi_amt_credit_adj, max(AMT_INCOME_MAIN) thu_nhap_lon_nhat
from CL_CONTRACT group by 1, 2 order by so_hd desc
''')

,so_dong,so_hop_dong
bang,,
CONTRACT gốc,6234093,4280377
CL_CONTRACT,4280377,4280377


,PRODUCT,PRODUCT_GROUP,so_hd,da_ky,trung_vi_amt_credit,trung_vi_amt_credit_adj,thu_nhap_lon_nhat
0,CD,CD,3095684,2109898.0,5593000.0,5593000.0,4.500000e+09
1,CL,CL,565521,377312.0,22000000.0,22000000.0,9.000000e+09
2,BNPL,BNPL,211481,178192.0,0.0,569050.0,9.000000e+08
3,CC,CC,135823,98597.0,19000000.0,19000000.0,9.000000e+08
4,TW,TW,133928,100793.0,21500000.0,21500000.0,5.000000e+08
5,CW,CW,109591,22397.0,20000000.0,20000000.0,9.500000e+08
6,None,None,28348,0.0,0.0,0.0,3.000000e+08
7,IN,IN,1,1.0,12000.0,12000.0,7.000000e+06


## 5. LEADS

Các thay đổi:

- Bỏ các dòng trùng hoàn toàn ở mọi cột.
- Giữ nguyên `DATE_ACCEPTED`, kể cả giá trị quy ước 3000-01-01 (không có lần bấm nhận lead). Khi tính khoảng thời gian từ lúc tạo đến lúc nhận lead, cần tự loại các dòng này.
- `MAX_OFFER` từ 1 đến 59 được hiểu là đơn vị triệu đồng và nhân với 1.000.000 (kể cả giá trị 1); giá trị 0 đổi thành trống.
- Thêm `PRODUCT_GROUP` theo bảng quy đổi ở mục 1.

In [ ]:
before = q('select count(*) so_dong from LEADS')
con.execute('''
CREATE OR REPLACE TABLE CL_LEADS AS
WITH d AS (SELECT DISTINCT * FROM LEADS)
SELECT d.DTIME_CREATED, d.DATE_ACCEPTED, d.DESC_FIN_REASON, d.CODE_PRODUCT_TYPE, d.FLAG_ASSIGNED, d.SKP_CLIENT,
       CASE WHEN d.MAX_OFFER = 0 THEN NULL
            WHEN d.MAX_OFFER BETWEEN 1 AND 59 THEN d.MAX_OFFER * 1000000
            ELSE d.MAX_OFFER END AS MAX_OFFER,
       d.LEAD_SOURCE, m.PRODUCT_GROUP
FROM d LEFT JOIN PRODUCT_MAP m ON d.CODE_PRODUCT_TYPE = m.CODE
''')
display(pd.concat([before.assign(bang='LEADS gốc'),
                   q('select count(*) so_dong from CL_LEADS').assign(bang='CL_LEADS')]).set_index('bang'))
q('''
select count(*) filter (where DATE_ACCEPTED >= TIMESTAMP '3000-01-01') date_accepted_3000,
       min(MAX_OFFER) max_offer_nho_nhat, count(*) filter (where MAX_OFFER < 100000) max_offer_duoi_100k,
       count(*) filter (where MAX_OFFER is null) max_offer_trong
from CL_LEADS
''')

,so_dong
bang,
LEADS gốc,2143891
CL_LEADS,2104642


,date_accepted_3000,max_offer_nho_nhat,max_offer_duoi_100k,max_offer_trong
0,578772,100000.0,0,12382


## 6. PAST_BEHAVIOR

Các thay đổi:

- Vài dòng gộp hai kênh trong cùng một ô, ví dụ "SA, TLS SCORING". Mỗi dòng như vậy được tách thành một dòng cho mỗi kênh ("SA SCORING" và "TLS SCORING").
- Thêm `PRODUCT_GROUP` theo bảng quy đổi ở mục 1.

In [ ]:
display(q("select ACTION, count(*) so_dong from PAST_BEHAVIOR where ACTION like '%,%' group by 1"))
con.execute('''
CREATE OR REPLACE TABLE CL_PAST_BEHAVIOR AS
WITH s AS (
  SELECT DTIME_EVENT, SKP_CLIENT, ACTION, PRODUCT_CODE FROM PAST_BEHAVIOR WHERE ACTION NOT LIKE '%,%'
  UNION ALL
  SELECT DTIME_EVENT, SKP_CLIENT, trim(unnest(string_split(replace(ACTION, ' SCORING', ''), ','))) || ' SCORING' AS ACTION, PRODUCT_CODE
  FROM PAST_BEHAVIOR WHERE ACTION LIKE '%,%'
)
SELECT s.*, m.PRODUCT_GROUP FROM s LEFT JOIN PRODUCT_MAP m ON s.PRODUCT_CODE = m.CODE
''')
display(q("select (select count(*) from PAST_BEHAVIOR) so_dong_goc, (select count(*) from CL_PAST_BEHAVIOR) so_dong_sau"))
q('select ACTION, count(*) so_dong from CL_PAST_BEHAVIOR group by 1 order by 2 desc')

,ACTION,so_dong
0,"APP, SA SCORING",2
1,"SA, WEB SCORING",1
2,"APP, Partner Site SCORING",1
3,"SA, TLS SCORING",3


,so_dong_goc,so_dong_sau
0,4412610,4412617


,ACTION,so_dong
0,APP SCORING,1630352
1,SA SCORING,867793
2,WEB SCORING,746022
3,TLS SCORING,726236
4,Partner Site SCORING,396830
5,SOB_CD_QR,37813
6,SOB_TW_LINK,5415
7,SOB_TW_QR,1136
8,TW0BOD,594
9,SOB_CD_LINK,258


## 7. CALL

Các thay đổi:

- Không bỏ dòng nào. `CALL_ID` bị dùng lại cho các cuộc gọi khác ngày, khác nội dung, nên mỗi dòng vẫn là một cuộc gọi.
- Thêm `FLAG_AFTER_SNAPSHOT = 1` cho các cuộc gọi sau ngày chốt 04/06/2025 (khoảng 75% số cuộc gọi). Nhóm chưa chốt có dùng phần này hay không, nên giữ lại và đánh dấu để mỗi người tự lọc.
- `PRODUCT_TYPE = XNA` đổi thành trống. Thêm `PRODUCT_GROUP` theo bảng quy đổi ở mục 1.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE CL_CALL AS
SELECT c.SKP_CLIENT, c.CALL_ID, c.CALL_TYPE, c.TIME_OF_CALL_START, c.TIME_OF_CALL_END, c.DURATIONS,
       CASE WHEN c.PRODUCT_TYPE = 'XNA' THEN NULL ELSE c.PRODUCT_TYPE END AS PRODUCT_TYPE,
       c.CALL_SOURCE, c.OFF_BY, c.OFFER_TYPE, c.RESPONSE_RESULT_2, c.CODE_DISPOSITION, c.CALL_RESULT_SYSTEM,
       CASE WHEN c.TIME_OF_CALL_START >= TIMESTAMP '{SNAPSHOT}' + INTERVAL 1 DAY THEN 1 ELSE 0 END AS FLAG_AFTER_SNAPSHOT,
       m.PRODUCT_GROUP
FROM CALL c LEFT JOIN PRODUCT_MAP m ON c.PRODUCT_TYPE = m.CODE
''')
q('''
select count(*) so_dong, sum(FLAG_AFTER_SNAPSHOT) sau_ngay_chot,
       round(100.0 * sum(FLAG_AFTER_SNAPSHOT) / count(*), 1) pct_sau_ngay_chot,
       count(*) filter (where PRODUCT_TYPE is null) product_type_trong
from CL_CALL
''')

,so_dong,sau_ngay_chot,pct_sau_ngay_chot,product_type_trong
0,4000000,2977789.0,74.4,1161694


## 8. Kiểm tra cuối và xuất file

Kiểm tra lại khóa nối: mọi khách ở 4 bảng còn lại đều phải có trong `CL_CUSTOMER`.

In [ ]:
q('''
select 'CL_CONTRACT' bang, count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CL_CUSTOMER)) khach_khong_co_trong_customer from CL_CONTRACT
union all select 'CL_LEADS', count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CL_CUSTOMER)) from CL_LEADS
union all select 'CL_PAST_BEHAVIOR', count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CL_CUSTOMER)) from CL_PAST_BEHAVIOR
union all select 'CL_CALL', count(distinct SKP_CLIENT) filter (where SKP_CLIENT not in (select SKP_CLIENT from CL_CUSTOMER)) from CL_CALL
''')

,bang,khach_khong_co_trong_customer
0,CL_CONTRACT,0
1,CL_LEADS,0
2,CL_PAST_BEHAVIOR,0
3,CL_CALL,0


Ba file có chữ tiếng Việt có dấu (`CUSTOMER_clean.csv`, `PRODUCT_MAPPING.csv`, `PROVINCE_MAPPING.csv`) được ghi thêm dấu hiệu UTF-8 ở đầu file. Nếu thiếu dấu hiệu này, Excel đọc sai bảng mã và hiện tên tỉnh thành ký tự lạ, ví dụ "Há»“ ChÃ­ Minh". pandas, DuckDB và Power BI vẫn đọc các file này bình thường.

In [ ]:
import shutil

EXPORTS = {
    'CL_CUSTOMER': 'CUSTOMER_clean.csv', 'CL_CONTRACT': 'CONTRACT_clean.csv', 'CL_LEADS': 'LEADS_clean.csv',
    'CL_PAST_BEHAVIOR': 'PAST_BEHAVIOR_clean.csv', 'CL_CALL': 'CALL_clean.csv',
    'PRODUCT_MAP': 'PRODUCT_MAPPING.csv', 'PROVINCE_MAP': 'PROVINCE_MAPPING.csv',
}
BOM_FILES = {'CUSTOMER_clean.csv', 'PRODUCT_MAPPING.csv', 'PROVINCE_MAPPING.csv'}
rows = []
for t, f in EXPORTS.items():
    path = f'{OUT_DIR}/{f}'
    if f in BOM_FILES:
        tmp = path + '.tmp'
        con.execute(f"COPY {t} TO '{tmp}' (HEADER, DELIMITER ',')")
        with open(tmp, 'rb') as src, open(path, 'wb') as dst:
            dst.write(b'\xef\xbb\xbf')
            shutil.copyfileobj(src, dst, 16 * 1024 * 1024)
        os.remove(tmp)
    else:
        con.execute(f"COPY {t} TO '{path}' (HEADER, DELIMITER ',')")
    rows.append({'file': f, 'so_dong': con.sql(f'select count(*) from {t}').fetchone()[0],
                 'so_cot': len(con.sql(f'select * from {t} limit 0').columns),
                 'dung_luong_MB': round(os.path.getsize(path) / 1e6, 1)})
pd.DataFrame(rows)

,file,so_dong,so_cot,dung_luong_MB
0,CUSTOMER_clean.csv,1000528,9,67.2
1,CONTRACT_clean.csv,4280377,28,853.5
2,LEADS_clean.csv,2104642,9,180.9
3,PAST_BEHAVIOR_clean.csv,4412617,5,220.6
4,CALL_clean.csv,4000000,15,548.3
5,PRODUCT_MAPPING.csv,21,2,0.0
6,PROVINCE_MAPPING.csv,68,2,0.0
